# The Ising Model: From Statistical Physics to Neural Networks

The **Ising model** is the simplest model of a magnetic material. It was proposed by Wilhelm Lenz in 1920 and solved analytically in 1D by his student Ernst Ising in 1925. Despite its simplicity, it exhibits **phase transitions** and connects directly to Hopfield networks and Boltzmann machines.

## The Model

A lattice of spins $s_i \in \{-1, +1\}$ with nearest-neighbor coupling:

$$E(\{s\}) = -J \sum_{\langle i,j \rangle} s_i s_j$$

where $J > 0$ favors alignment (ferromagnetic). At temperature $T$, configurations follow the **Boltzmann distribution**:

$$P(\{s\}) = \frac{1}{Z} e^{-E(\{s\})/T}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# Grid size
N = 50

def init_grid(N, p=0.5):
    """Random initial spin configuration."""
    return np.where(np.random.rand(N, N) < p, 1, -1)

def ising_step(grid, T, J=1.0):
    """One Monte Carlo sweep: N^2 random single-spin updates."""
    N = grid.shape[0]
    for _ in range(N * N):
        i, j = np.random.randint(N, size=2)
        s = grid[i, j]
        # Sum of 4 nearest neighbors (periodic boundaries)
        neighbors = (grid[(i-1)%N, j] + grid[(i+1)%N, j] +
                     grid[i, (j-1)%N] + grid[i, (j+1)%N])
        dE = 2 * J * s * neighbors
        # Metropolis criterion
        if dE <= 0 or np.random.rand() < np.exp(-dE / T):
            grid[i, j] = -s
    return grid

## Phase Transition

The 2D Ising model has a **critical temperature** $T_c \approx 2.27$ (in units of $J/k_B$).

- **$T < T_c$**: Ordered phase — most spins align (spontaneous magnetization)
- **$T > T_c$**: Disordered phase — spins point randomly (paramagnetic)
- **$T = T_c$**: Critical point — fluctuations at all scales, power-law correlations

Let's simulate the Ising model at three temperatures and watch the dynamics.

In [ ]:
# Simulate at three temperatures
temperatures = [1.5, 2.27, 4.0]
labels = [f'T = {T} ({name})' for T, name in 
          zip(temperatures, ['ordered', 'critical', 'disordered'])]
n_steps = 100

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

for ax, T, label in zip(axes, temperatures, labels):
    grid = init_grid(N)
    # Equilibrate
    for _ in range(n_steps):
        grid = ising_step(grid, T)
    ax.imshow(grid, cmap='coolwarm', vmin=-1, vmax=1, interpolation='nearest')
    ax.set_title(label, fontsize=12)
    ax.axis('off')

plt.suptitle('Ising Model After 100 Monte Carlo Sweeps', fontsize=14)
plt.tight_layout()
plt.show()

## Measuring the Magnetization

The **order parameter** is the average magnetization:

$$m = \frac{1}{N^2} \left| \sum_i s_i \right|$$

Near $T_c$, it drops from $m \approx 1$ to $m \approx 0$. Let's trace $m(T)$.

In [ ]:
# Sweep temperature and measure magnetization
T_range = np.linspace(1.0, 4.0, 25)
mag_avg = []
n_equil = 50   # equilibration sweeps
n_sample = 30  # measurement sweeps

for T in T_range:
    grid = init_grid(N)
    # Equilibrate
    for _ in range(n_equil):
        grid = ising_step(grid, T)
    # Measure
    mags = []
    for _ in range(n_sample):
        grid = ising_step(grid, T)
        mags.append(np.abs(grid.mean()))
    mag_avg.append(np.mean(mags))

plt.figure(figsize=(8, 4))
plt.plot(T_range, mag_avg, 'o-', color='#61afef', markersize=5)
plt.axvline(2.27, color='#e06c75', linestyle='--', label=r'$T_c \approx 2.27$')
plt.xlabel('Temperature $T$', fontsize=12)
plt.ylabel(r'Magnetization $\langle |m| \rangle$', fontsize=12)
plt.title('Phase Transition in the 2D Ising Model', fontsize=13)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.show()

## Connection to Neural Networks

The Ising model is the direct ancestor of:

| Physics | Neural Networks |
|---------|----------------|
| Spin $s_i \in \{-1, +1\}$ | Neuron activation |
| Coupling $J_{ij}$ | Synaptic weight $w_{ij}$ |
| Energy $E(\{s\})$ | Loss / cost function |
| Boltzmann dist. $e^{-E/T}$ | Softmax / Gibbs sampling |
| Energy minimization | Learning |

**Hopfield (1982)**: Replace the lattice with a fully connected graph. Local energy minima become **stored memories**.

**Boltzmann machines (1985)**: Add hidden units. Train by adjusting $J_{ij}$ to match data statistics.

The mathematical framework is identical — only the interpretation changes.

## Exercises

1. **Critical slowing down**: Measure how many sweeps it takes to reach equilibrium at $T = T_c$ vs. $T = 1.5$. Why is it slower near the critical point?

2. **Energy vs. temperature**: Plot the average energy $\langle E \rangle$ as a function of $T$. Compute the **specific heat** $C = \partial \langle E \rangle / \partial T$ and find its peak near $T_c$.

3. **External field**: Add an external magnetic field $h$ to the energy: $E = -J \sum s_i s_j - h \sum s_i$. How does the magnetization curve change?

4. **Hopfield network**: Modify the code so that $J_{ij}$ stores patterns via the Hebbian rule $J_{ij} = \frac{1}{P}\sum_{\mu=1}^P \xi_i^\mu \xi_j^\mu$. Initialize the grid near a stored pattern and watch it converge.